In [1]:
%load_ext autoreload
%autoreload 2

from seek_code import SEEK
from backbone import Backbone
from concept_head import ConceptHead
from retrieval import Retrieval
from projector import Projector
from data_handler import EleHandler
from concept_head_tunneled import ConceptHeadTunneled, categorical_CE_loss, CrossedHeadNN, MultiHeadNN


from torch.utils.data import DataLoader, Subset
import torch.nn.functional as F
import torch.nn as nn

/data/vision/beery/scratch/antoine/micromamba/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Replicating previous results (76.5%)

In [18]:
code = 'MultiView-v2'

dataset = EleHandler(subset='IDI_6')
train_indices, test_indices = dataset.split_perpendicular_to_elephants_and_encounters(split_sizes=[0.5,0.5], hour_delta = 0.2)

train_subset = Subset(dataset, train_indices)
test_subset = Subset(dataset, test_indices)

train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=64, shuffle=True)

In [19]:
MD_finetuned = Backbone(model_name="MegaDescriptor", pretraining="backbone_for_concepts_w", experiment_code=code)
MD_finetuned.unfreeze()

concept_cat = ConceptHeadTunneled(loss=categorical_CE_loss, print_every=1, experiment_code = code, layer = CrossedHeadNN(), reset_weights=False)

concept_cat.train(train_loader, test_loader, backbone = MD_finetuned, num_epochs=25, predict_ele_SEEK = False)
concept_cat.test(test_loader, backbone = MD_finetuned, predict_ele_SEEK = False)

r = Retrieval(experiment_code=code)

r.evaluate_model(concept_cat, ba=MD_finetuned, train_loader=train_loader, test_loader=test_loader, show_matches=False, show_plot=False, show_tsne=False)


/data/vision/beery/scratch/antoine/CBM_reid/methods/backbone.py:33: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(weight_path, map_location=self.devi

loading weights for the concept head
Training ConceptHead for 25 epochs, reseting weights, and the backbone parameters are, True  concept head parameters are  True
Epoch 1/25 - Train Loss: 20.8032, Val Loss: 18.9543 - Train Acc: 36.39%, Val Acc: 57.12% - Train whole-code: 0.00%, Val whole-code: 0.00% - Train left ear: 29.38%, Val left ear: 46.89% - Train right ear: 24.36%, Val right ear: 36.58%
Epoch 2/25 - Train Loss: 17.1724, Val Loss: 14.5128 - Train Acc: 59.33%, Val Acc: 66.60% - Train whole-code: 0.00%, Val whole-code: 0.00% - Train left ear: 48.42%, Val left ear: 55.17% - Train right ear: 40.89%, Val right ear: 52.95%
Epoch 3/25 - Train Loss: 14.4061, Val Loss: 13.2377 - Train Acc: 65.73%, Val Acc: 67.32% - Train whole-code: 0.10%, Val whole-code: 0.22% - Train left ear: 53.96%, Val left ear: 56.17% - Train right ear: 51.27%, Val right ear: 54.43%
Epoch 4/25 - Train Loss: 13.6068, Val Loss: 12.8750 - Train Acc: 66.41%, Val Acc: 68.49% - Train whole-code: 0.20%, Val whole-code: 0.

100%|██████████| 31/31 [00:20<00:00,  1.53it/s]


collecting concepts from concept head


100%|██████████| 21/21 [00:13<00:00,  1.60it/s]


Train Recalls:
Recall@1: 0.61%
Recall@5: 2.42%
Recall@10: 4.54%
Recall@20: 8.38%
Recall@100: 32.26%
Test Recalls:
Recall@1: 0.69%
Recall@5: 3.50%
Recall@10: 6.25%
Recall@20: 10.13%
Recall@100: 38.31%


## Performing aggregation across all elephants (not limited to encounters)

In [22]:
# Apply the function to get the final structured DataFrame with SEEK codes
SEEK_codes, ele_ids = concept_cat.collect_embeddings(test_loader, MD_finetuned, aggregate_seeks=False)
subject_seek, ele_seek, ele_id = SEEK.aggregate_seek(SEEK_codes, ele_ids)

import pandas as pd
df = pd.DataFrame({
    'subject_seek': [SEEK(element).__str__() for element in subject_seek],
    'ele_seek': [SEEK(element).__str__() for element in ele_seek],
    'ele_id': ele_id
})
print(df.head(50))

collecting concepts from concept head


 76%|███████▌  | 16/21 [00:10<00:03,  1.51it/s]

In [20]:
#concept_cat.test(test_loader, backbone = MD_finetuned, predict_ele_SEEK = False)
# Convert the mappings to a DataFrame
r = Retrieval(experiment_code=code)
r.evaluate_model(concept_cat, ba=MD_finetuned, train_loader=train_loader, test_loader=test_loader, show_matches=False, show_plot=False, show_tsne=False, aggregate_seeks=True)

evaluating concept head  
collecting concepts from concept head


100%|██████████| 31/31 [00:20<00:00,  1.51it/s]


collecting concepts from concept head


100%|██████████| 21/21 [00:14<00:00,  1.49it/s]


Train Recalls:
Recall@1: 8.78%
Recall@5: 20.95%
Recall@10: 33.17%
Recall@20: 45.48%
Recall@100: 82.28%
Test Recalls:
Recall@1: 0.23%
Recall@5: 2.13%
Recall@10: 5.79%
Recall@20: 11.42%
Recall@100: 25.82%


In [24]:
r = Retrieval(experiment_code=code)
print('------------retrieval on extracted subject SEEK codes----------------')
r.evaluate_model(concept_cat, ba=MD_finetuned, train_loader=train_loader, test_loader=test_loader, show_tsne = False, show_matches=False, show_plot=False, aggregate_seeks=False, intervention_fn=None)
print('------------retrieval on aggregated extracted subject SEEK codes----------------')
r.evaluate_model(concept_cat, ba=MD_finetuned, train_loader=train_loader, test_loader=test_loader, show_tsne = False, show_matches=False, show_plot=False, aggregate_seeks=True, intervention_fn=None)
print('------------retrieval on perfect subject SEEK codes----------------')
r.evaluate_model(concept_cat, ba=MD_finetuned, train_loader=train_loader, test_loader=test_loader, show_tsne = False, show_matches=False, show_plot=False, aggregate_seeks=False, intervention_fn=perfect_correction)
print('------------retrieval on aggregated perfect subject SEEK codes----------------')
r.evaluate_model(concept_cat, ba=MD_finetuned, train_loader=train_loader, test_loader=test_loader, show_tsne = False, show_matches=False, show_plot=False, aggregate_seeks=True, intervention_fn=perfect_correction)
print('------------retrieval on elephant SEEK codes----------------')
r.evaluate_model(concept_cat, ba=MD_finetuned, train_loader=train_loader, test_loader=test_loader, show_tsne = False, show_matches=False, show_plot=False, aggregate_seeks=False, intervention_fn=oracle_correction)

------------retrieval on extracted subject SEEK codes----------------
evaluating concept head  
collecting concepts from concept head


100%|██████████| 31/31 [00:18<00:00,  1.71it/s]


collecting concepts from concept head


100%|██████████| 21/21 [00:12<00:00,  1.73it/s]


Train Recalls:
Recall@1: 0.86%
Recall@5: 2.42%
Recall@10: 4.19%
Recall@20: 7.77%
Recall@100: 33.47%
Test Recalls:
Recall@1: 0.69%
Recall@5: 2.36%
Recall@10: 5.26%
Recall@20: 9.06%
Recall@100: 38.61%
------------retrieval on aggregated extracted subject SEEK codes----------------
evaluating concept head  
collecting concepts from concept head


...

Train Recalls:
Recall@1: 81.93%
Recall@5: 96.72%
Recall@10: 99.24%
Recall@20: 99.95%
Recall@100: 100.00%
Test Recalls:
Recall@1: 11.04%
Recall@5: 15.38%
Recall@10: 20.64%
Recall@20: 27.19%
Recall@100: 50.88%
------------retrieval on elephant SEEK codes----------------
evaluating concept head   with intervention
collecting concepts from concept head


100%|██████████| 31/31 [00:18<00:00,  1.66it/s]


collecting concepts from concept head


100%|██████████| 21/21 [00:12<00:00,  1.72it/s]


Train Recalls:
Recall@1: 79.61%
Recall@5: 95.20%
Recall@10: 98.18%
Recall@20: 100.00%
Recall@100: 100.00%
Test Recalls:
Recall@1: 80.73%
Recall@5: 95.96%
Recall@10: 98.48%
Recall@20: 100.00%
Recall@100: 100.00%


## Bon vu que c'est le souk. Je vais essayer d'apprendre les ele_SEEK, puis aggregate. 

In [ ]:
code = 'Multiview_on_ele_SEEK'

train_loader = DataLoader(train_subset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_subset, batch_size=64, shuffle=False)

ele_MD_finetuned = Backbone(model_name="MegaDescriptor", pretraining="MD_finetuned_w", experiment_code=code)
ele_MD_finetuned.unfreeze()

ele_concept = ConceptHeadTunneled(lr = 1e-5, loss=categorical_CE_loss, print_every=1, experiment_code = code, layer = CrossedHeadNN(), reset_weights=True)

ele_concept.train(train_loader, test_loader, backbone = ele_MD_finetuned, num_epochs=50, predict_ele_SEEK = True)
ele_concept.test(test_loader, backbone = ele_MD_finetuned, predict_ele_SEEK = True)



/data/vision/beery/scratch/antoine/CBM_reid/methods/backbone.py:33: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(weight_path, map_location=self.devi

Training ConceptHead for 50 epochs, reseting weights, and the backbone parameters are, True  concept head parameters are  True


OutOfMemoryError: CUDA out of memory. Tried to allocate 106.00 MiB. GPU 0 has a total capacity of 79.14 GiB of which 27.19 MiB is free. Process 2669040 has 616.00 MiB memory in use. Including non-PyTorch memory, this process has 78.50 GiB memory in use. Of the allocated memory 73.57 GiB is allocated by PyTorch, and 4.42 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
def perfect_correction(hard_predicted_concepts, subject_SEEK, elephant_SEEK):
    return subject_SEEK

def oracle_correction(hard_predicted_concepts, subject_SEEK, elephant_SEEK):
    return elephant_SEEK

extracted_ele_SEEK_codes, ele_ids_1 = ele_concept.collect_embeddings(test_loader, ele_MD_finetuned, aggregate_seeks=False, intervention_fn=None)
aggregated_ele_SEEK_codes, ele_ids_2 = ele_concept.collect_embeddings(test_loader, ele_MD_finetuned, aggregate_seeks=True, intervention_fn=None)
perfect_ele_SEEK_codes, ele_ids_5 = ele_concept.collect_embeddings(test_loader, ele_MD_finetuned, aggregate_seeks=False, intervention_fn=oracle_correction)

import pandas as pd
df = pd.DataFrame({
    'extracted_ele_SEEK_codes': [SEEK(element).__str__() for element in extracted_ele_SEEK_codes],
    'aggregated_ele_SEEK_codes': [SEEK(element).__str__() for element in aggregated_ele_SEEK_codes],
    'perfect_ele_SEEK_codes': [SEEK(element).__str__() for element in perfect_ele_SEEK_codes],
    'ele_id_1': ele_ids_1.cpu(),
    'ele_id_2': ele_ids_2.cpu(),
    'ele_id_5': ele_ids_5.cpu()
})
df

collecting concepts from concept head


100%|██████████| 15/15 [00:08<00:00,  1.72it/s]


collecting concepts from concept head


100%|██████████| 15/15 [00:08<00:00,  1.74it/s]


collecting concepts from concept head


100%|██████████| 15/15 [00:08<00:00,  1.69it/s]


In [ ]:
r = Retrieval(experiment_code=code)
print('------------retrieval on extracted ele SEEK codes----------------')
r.evaluate_model(ele_concept, ba=ele_MD_finetuned, train_loader=train_loader, test_loader=test_loader, show_tsne = False, show_matches=False, show_plot=False, aggregate_seeks=False, intervention_fn=None)
print('------------retrieval on aggregated extracted ele SEEK codes----------------')
r.evaluate_model(ele_concept, ba=ele_MD_finetuned, train_loader=train_loader, test_loader=test_loader, show_tsne = False, show_matches=False, show_plot=False, aggregate_seeks=True, intervention_fn=None)
print('------------retrieval on perfect elephant SEEK codes----------------')
r.evaluate_model(ele_concept, ba=ele_MD_finetuned, train_loader=train_loader, test_loader=test_loader, show_tsne = False, show_matches=False, show_plot=False, aggregate_seeks=False, intervention_fn=oracle_correction)

------------retrieval on extracted ele SEEK codes----------------
evaluating concept head  
collecting concepts from concept head


100%|██████████| 22/22 [00:14<00:00,  1.54it/s]


collecting concepts from concept head


100%|██████████| 15/15 [00:08<00:00,  1.76it/s]


Train Recalls:
Recall@1: 67.54%
Recall@5: 86.42%
Recall@10: 92.63%
Recall@20: 96.72%
Recall@100: 99.60%
Test Recalls:
Recall@1: 1.98%
Recall@5: 3.81%
Recall@10: 4.87%
Recall@20: 6.32%
Recall@100: 16.83%
------------retrieval on aggregated extracted ele SEEK codes----------------
evaluating concept head  
collecting concepts from concept head


...

Train Recalls:
Recall@1: 79.20%
Recall@5: 93.89%
Recall@10: 97.53%
Recall@20: 99.70%
Recall@100: 100.00%
Test Recalls:
Recall@1: 1.22%
Recall@5: 2.06%
Recall@10: 3.50%
Recall@20: 4.04%
Recall@100: 14.70%
------------retrieval on perfect elephant SEEK codes----------------
evaluating concept head   with intervention
collecting concepts from concept head


100%|██████████| 22/22 [00:13<00:00,  1.64it/s]


collecting concepts from concept head


100%|██████████| 15/15 [00:08<00:00,  1.79it/s]


Train Recalls:
Recall@1: 81.17%
Recall@5: 94.04%
Recall@10: 98.13%
Recall@20: 99.95%
Recall@100: 100.00%
Test Recalls:
Recall@1: 82.48%
Recall@5: 94.97%
Recall@10: 98.48%
Recall@20: 100.00%
Recall@100: 100.00%
